# Generate Initial Tennis Ball Coordinates

This notebook converts TrackNet's 256-class heatmap into a dense ball-probability map, clusters every foreground region, and exports all candidate coordinates per frame.

Each output filename starts with the uploaded video's filename (without its extension). For example, `match01.mp4` produces:
- `match01_ball_coords_raw.json`
- `match01_ball_coords_match_frames.json`
- `match01_ball_coords_raw_overlay.mp4`
- `match01_ball_coords_trajectory.png`

In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "experiments"} and PROJECT_ROOT.parent.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cv.config import DEFAULT_BALL_MODEL_PATH, INPUT_DIR, OUTPUT_DIR as CV_OUTPUT_DIR
from cv.models.ball_tracking.model import BallTrackerNet

## 1. Configure Paths

Set `INPUT_VIDEO_PATH` to the uploaded video. Output names are derived automatically from this filename.

In [ ]:
INPUT_VIDEO_PATH = INPUT_DIR / "real_court/real02.mp4"

# Override this manually if your weights live elsewhere.
CANDIDATE_BALL_MODEL_PATHS = [
    DEFAULT_BALL_MODEL_PATH,
    PROJECT_ROOT / "weights/model_best.pt",
]
BALL_MODEL_PATH = next(
    (path for path in CANDIDATE_BALL_MODEL_PATHS if path.exists()),
    CANDIDATE_BALL_MODEL_PATHS[0],
)

OUTPUT_DIR = CV_OUTPUT_DIR
OUTPUT_PREFIX = INPUT_VIDEO_PATH.stem
RAW_OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_raw.json"
MATCH_FRAMES_OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_match_frames.json"
RAW_POINTS_VIDEO_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_raw_overlay.mp4"
TRAJECTORY_PLOT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_trajectory.png"

# Set to an integer for a shorter test run, or None for the whole video.
MAX_FRAMES = None

# TrackNet predicts one of 256 heatmap-intensity classes at every pixel.
# P(class >= BALLNESS_CLASS_THRESHOLD) is treated as pixel-level ball confidence.
BALLNESS_CLASS_THRESHOLD = 128
MIN_PIXEL_BALL_PROBABILITY = 0.15
RELATIVE_PIXEL_THRESHOLD = 0.35
MIN_CLUSTER_PIXELS = 2
CLUSTER_CONNECTIVITY = 8

print("video:", INPUT_VIDEO_PATH)
print("ball model:", BALL_MODEL_PATH)
print("raw output:", RAW_OUTPUT_PATH)
print("match_frames output:", MATCH_FRAMES_OUTPUT_PATH)
print("overlay video:", RAW_POINTS_VIDEO_PATH)
print("trajectory plot:", TRAJECTORY_PLOT_PATH)

In [ ]:
if not INPUT_VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO_PATH}")
if not BALL_MODEL_PATH.exists():
    raise FileNotFoundError(
        "TrackNet ball model not found. Set BALL_MODEL_PATH to model_best.pt before running. "
        f"Current value: {BALL_MODEL_PATH}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Model

The exported pixel coordinates use the source video's original resolution.

In [ ]:
def choose_torch_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = choose_torch_device()
model = BallTrackerNet()
state_dict = torch.load(BALL_MODEL_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("device:", device)

## 3. Extract Initial Coordinates

For every frame, `candidates` contains one probability-weighted center per connected heatmap cluster. `pixel` remains as a backward-compatible alias for the highest-confidence candidate. No temporal path selection, interpolation, or trajectory cleanup is applied.

In [ ]:
def cluster_tracknet_heatmap(probs, *, orig_w, orig_h):
    n_classes, input_h, input_w = probs.shape
    ball_probability = probs[BALLNESS_CLASS_THRESHOLD:].sum(dim=0).detach().cpu().numpy()
    class_intensity = torch.linspace(0.0, 1.0, n_classes, device=probs.device)[:, None, None]
    expected_heat = (probs * class_intensity).sum(dim=0).detach().cpu().numpy()

    frame_peak_probability = float(ball_probability.max())
    pixel_threshold = max(
        MIN_PIXEL_BALL_PROBABILITY,
        frame_peak_probability * RELATIVE_PIXEL_THRESHOLD,
    )
    foreground = (ball_probability >= pixel_threshold).astype(np.uint8)
    component_count, labels, stats, _ = cv2.connectedComponentsWithStats(
        foreground,
        connectivity=CLUSTER_CONNECTIVITY,
    )

    scale_x = orig_w / input_w
    scale_y = orig_h / input_h
    candidates = []
    for component_id in range(1, component_count):
        ys, xs = np.where(labels == component_id)
        area = int(xs.size)
        if area < MIN_CLUSTER_PIXELS:
            continue

        cluster_probability = ball_probability[ys, xs]
        cluster_expected_heat = expected_heat[ys, xs]
        weights = cluster_probability * np.maximum(cluster_expected_heat, 1e-6)
        weight_sum = float(weights.sum())
        if weight_sum <= 0:
            continue

        center_x = float(np.average(xs, weights=weights))
        center_y = float(np.average(ys, weights=weights))
        top_k = min(5, area)
        strongest_probabilities = np.partition(cluster_probability, -top_k)[-top_k:]
        confidence = float(strongest_probabilities.mean())

        left = int(stats[component_id, cv2.CC_STAT_LEFT])
        top = int(stats[component_id, cv2.CC_STAT_TOP])
        width = int(stats[component_id, cv2.CC_STAT_WIDTH])
        height = int(stats[component_id, cv2.CC_STAT_HEIGHT])
        candidates.append({
            "pixel": [
                round((center_x + 0.5) * scale_x, 2),
                round((center_y + 0.5) * scale_y, 2),
            ],
            "confidence": round(confidence, 4),
            "peak_confidence": round(float(cluster_probability.max()), 4),
            "mean_confidence": round(float(cluster_probability.mean()), 4),
            "heat_mass": round(weight_sum, 4),
            "cluster_pixels": area,
            "bbox": [
                round(left * scale_x, 2),
                round(top * scale_y, 2),
                round(width * scale_x, 2),
                round(height * scale_y, 2),
            ],
        })

    candidates.sort(key=lambda item: (item["confidence"], item["heat_mass"]), reverse=True)
    for rank, candidate in enumerate(candidates, start=1):
        candidate["rank"] = rank

    return {
        "candidates": candidates,
        "frame_peak_probability": round(frame_peak_probability, 4),
        "pixel_probability_threshold": round(float(pixel_threshold), 4),
    }


def infer_ball_candidates(model, frames_3, *, device, orig_w, orig_h):
    f2, f1, f0 = frames_3  # Oldest, middle, newest; matches the existing TrackNet pipeline order.
    input_w, input_h = 640, 360
    img2 = cv2.resize(f2, (input_w, input_h))
    img1 = cv2.resize(f1, (input_w, input_h))
    img0 = cv2.resize(f0, (input_w, input_h))

    imgs = np.concatenate((img0, img1, img2), axis=2).astype(np.float32) / 255.0
    imgs = np.rollaxis(imgs, 2, 0)
    inp = np.expand_dims(imgs, axis=0)

    with torch.no_grad():
        logits = model(torch.from_numpy(inp).float().to(device))

    n_classes = int(logits.shape[1])
    probs = torch.softmax(logits.reshape(1, n_classes, input_h, input_w), dim=1)[0]
    return cluster_tracknet_heatmap(
        probs,
        orig_w=orig_w,
        orig_h=orig_h,
    )


cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO_PATH}")

fps = float(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
target_frames = min(total_frames, MAX_FRAMES) if MAX_FRAMES is not None else total_frames

ball_coords = []
frame_buffer = deque(maxlen=3)
progress = tqdm(total=target_frames, desc="Extract initial TrackNet coordinates")

for frame_index in range(target_frames):
    ok, frame = cap.read()
    if not ok:
        break

    frame_buffer.append(frame.copy())
    detection = {
        "candidates": [],
        "frame_peak_probability": None,
        "pixel_probability_threshold": None,
    }
    if len(frame_buffer) == 3:
        detection = infer_ball_candidates(
            model,
            list(frame_buffer),
            device=device,
            orig_w=frame_width,
            orig_h=frame_height,
        )

    candidates = detection["candidates"]
    primary = candidates[0] if candidates else None
    ball_coords.append({
        "frame_index": frame_index,
        "time_sec": round(frame_index / fps, 3) if fps > 0 else 0.0,
        "pixel": primary["pixel"] if primary else None,
        "confidence": primary["confidence"] if primary else None,
        "candidates": candidates,
        "candidate_count": len(candidates),
        "frame_peak_probability": detection["frame_peak_probability"],
        "pixel_probability_threshold": detection["pixel_probability_threshold"],
        "coordinate_method": "tracknet_probability_connected_components",
    })
    progress.update(1)

cap.release()
progress.close()

detected = sum(item["candidate_count"] > 0 for item in ball_coords)
multi_candidate_frames = sum(item["candidate_count"] > 1 for item in ball_coords)
all_candidates = [candidate for item in ball_coords for candidate in item["candidates"]]
valid_confidences = [candidate["confidence"] for candidate in all_candidates]
print(
    f"frames={len(ball_coords):,}, frames_with_candidates={detected:,}, "
    f"total_candidates={len(all_candidates):,}, multi_candidate_frames={multi_candidate_frames:,}"
)

## 4. Plot Coordinate Functions and Trajectory

The left plots show every clustered candidate in `x(t)` and `y(t)`. The right plot shows all candidates in video coordinates, colored by time and sized by confidence. Candidates are not connected across frames because no temporal identity has been selected yet.

In [ ]:
candidate_times = np.array(
    [item["time_sec"] for item in ball_coords for _ in item["candidates"]],
    dtype=np.float64,
)
candidate_pixels = np.array(
    [candidate["pixel"] for item in ball_coords for candidate in item["candidates"]],
    dtype=np.float64,
).reshape(-1, 2)
candidate_confidences = np.array(
    [candidate["confidence"] for item in ball_coords for candidate in item["candidates"]],
    dtype=np.float64,
)

fig = plt.figure(figsize=(15, 8), constrained_layout=True)
grid = fig.add_gridspec(2, 2, width_ratios=(1.5, 1.0))
ax_x = fig.add_subplot(grid[0, 0])
ax_y = fig.add_subplot(grid[1, 0], sharex=ax_x)
ax_candidates = fig.add_subplot(grid[:, 1])

if len(candidate_pixels):
    marker_sizes = 10.0 + 38.0 * candidate_confidences
    ax_x.scatter(candidate_times, candidate_pixels[:, 0], c=candidate_confidences, cmap="plasma", s=8, alpha=0.75)
    ax_y.scatter(candidate_times, candidate_pixels[:, 1], c=candidate_confidences, cmap="plasma", s=8, alpha=0.75)
    points = ax_candidates.scatter(
        candidate_pixels[:, 0],
        candidate_pixels[:, 1],
        c=candidate_times,
        cmap="viridis",
        s=marker_sizes,
        alpha=0.75,
        edgecolors="none",
    )
    fig.colorbar(points, ax=ax_candidates, label="time (s)")
else:
    ax_candidates.text(0.5, 0.5, "No candidates detected", ha="center", va="center", transform=ax_candidates.transAxes)

ax_x.set_ylabel("x (pixel)")
ax_y.set_ylabel("y (pixel)")
ax_y.set_xlabel("time (s)")
ax_x.set_title("All candidate x coordinates")
ax_y.set_title("All candidate y coordinates")
ax_x.grid(alpha=0.25)
ax_y.grid(alpha=0.25)

ax_candidates.set_xlim(0, frame_width)
ax_candidates.set_ylim(frame_height, 0)
ax_candidates.set_aspect("equal", adjustable="box")
ax_candidates.set_xlabel("x (pixel)")
ax_candidates.set_ylabel("y (pixel)")
ax_candidates.set_title("All clustered candidates in video coordinates")
ax_candidates.grid(alpha=0.2)

fig.suptitle(f"TrackNet heatmap candidates: {INPUT_VIDEO_PATH.name}", fontsize=14)
fig.savefig(TRAJECTORY_PLOT_PATH, dpi=160, bbox_inches="tight")
print("wrote candidate plot:", TRAJECTORY_PLOT_PATH)
plt.show()

## 5. Render Initial Coordinate Video

This video draws every clustered candidate in each frame. Marker color and label show cluster confidence; no temporal tail is drawn because candidates have not been associated across frames.

In [ ]:
def render_initial_coordinate_overlay(
    input_video_path,
    output_video_path,
    ball_coords,
    *,
    fps,
    frame_width,
    frame_height,
    ball_radius=7,
):
    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video for rendering: {input_video_path}")

    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps if fps > 0 else 25.0,
        (frame_width, frame_height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open video writer: {output_video_path}")

    candidates_by_frame = {item["frame_index"]: item["candidates"] for item in ball_coords}
    total_count = len(ball_coords)
    detected_count = sum(bool(candidates) for candidates in candidates_by_frame.values())
    total_candidates = sum(len(candidates) for candidates in candidates_by_frame.values())
    rendered_count = 0
    progress = tqdm(total=total_count, desc="Render initial coordinate overlay")

    for frame_index in range(total_count):
        ok, frame = cap.read()
        if not ok:
            break
        canvas = frame.copy()

        frame_candidates = candidates_by_frame.get(frame_index, [])
        for candidate in frame_candidates:
            x, y = (int(round(value)) for value in candidate["pixel"])
            confidence = float(candidate["confidence"])
            color = (0, int(round(255 * confidence)), int(round(255 * (1.0 - confidence))))
            cv2.circle(canvas, (x, y), ball_radius + 3, (0, 0, 0), -1, cv2.LINE_AA)
            cv2.circle(canvas, (x, y), ball_radius, color, -1, cv2.LINE_AA)
            cv2.drawMarker(canvas, (x, y), (255, 255, 255), cv2.MARKER_CROSS, 12, 1, cv2.LINE_AA)
            label = f"#{candidate['rank']} {confidence:.3f}"
            label_x = min(max(4, x + ball_radius + 5), max(4, frame_width - 120))
            label_y = min(max(18, y - ball_radius - 4), frame_height - 6)
            cv2.putText(canvas, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (0, 0, 0), 3, cv2.LINE_AA)
            cv2.putText(canvas, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.46, color, 1, cv2.LINE_AA)

        hud_lines = [
            f"frame {frame_index}/{total_count - 1}",
            f"candidates this frame: {len(frame_candidates)}",
            f"frames with candidates: {detected_count}/{total_count} ({detected_count / max(total_count, 1):.1%})",
            f"total candidates: {total_candidates}",
        ]
        for line_index, text in enumerate(hud_lines):
            y_pos = 32 + line_index * 26
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA)

        writer.write(canvas)
        rendered_count += 1
        progress.update(1)

    cap.release()
    writer.release()
    progress.close()
    return rendered_count


rendered_frames = render_initial_coordinate_overlay(
    INPUT_VIDEO_PATH,
    RAW_POINTS_VIDEO_PATH,
    ball_coords,
    fps=fps,
    frame_width=frame_width,
    frame_height=frame_height,
)
print(f"wrote overlay video: {RAW_POINTS_VIDEO_PATH} ({rendered_frames} frames)")

## 6. Write JSON Outputs

In [ ]:
metadata = {
    "version": "ball-candidates-2.0.0",
    "source_video": INPUT_VIDEO_PATH.name,
    "source_video_path": str(INPUT_VIDEO_PATH),
    "output_prefix": OUTPUT_PREFIX,
    "ball_model_path": str(BALL_MODEL_PATH),
    "coordinate_method": "tracknet_probability_connected_components",
    "confidence_definition": "mean of the five strongest P(heatmap_class >= threshold) pixels in each cluster",
    "clustering": {
        "algorithm": "connected_components",
        "connectivity": CLUSTER_CONNECTIVITY,
        "ballness_class_threshold": BALLNESS_CLASS_THRESHOLD,
        "min_pixel_ball_probability": MIN_PIXEL_BALL_PROBABILITY,
        "relative_pixel_threshold": RELATIVE_PIXEL_THRESHOLD,
        "min_cluster_pixels": MIN_CLUSTER_PIXELS,
    },
    "fps": fps,
    "video_width": frame_width,
    "video_height": frame_height,
    "total_frames": len(ball_coords),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

raw_payload = {
    "metadata": metadata,
    "ball_coords": ball_coords,
    "summary": {
        "detected_frames": detected,
        "missing_frames": len(ball_coords) - detected,
        "detection_ratio": round(detected / max(len(ball_coords), 1), 4),
        "coordinate_method": "tracknet_probability_connected_components",
        "total_candidates": len(all_candidates),
        "multi_candidate_frames": multi_candidate_frames,
        "mean_candidates_per_detected_frame": round(len(all_candidates) / max(detected, 1), 4),
        "confidence_mean": round(float(np.mean(valid_confidences)), 4) if valid_confidences else None,
    },
}


def ball_payload_from_coord_item(item):
    if item["pixel"] is None:
        return None
    return {
        "pixel": item["pixel"],
        "court": None,
        "confidence": item["confidence"],
        "coordinate_method": item["coordinate_method"],
    }


def candidate_payloads_from_coord_item(item):
    return [
        {
            **candidate,
            "court": None,
            "coordinate_method": item["coordinate_method"],
        }
        for candidate in item["candidates"]
    ]


match_frames_payload = {
    "metadata": {
        **metadata,
        "version": "2.1.0-ball-candidates",
        "court_dimensions": {"unit": "meters", "width": 10.97, "length": 23.77},
    },
    "frames": [
        {
            "frame_index": item["frame_index"],
            "time_sec": item["time_sec"],
            "court_visible": False,
            "court_keypoints_detected": 0,
            "ball": ball_payload_from_coord_item(item),
            "ball_candidates": candidate_payloads_from_coord_item(item),
            "players": [],
            "events": {
                "is_bounce": False,
                "is_hit": False,
                "is_net_hit": False,
            },
        }
        for item in ball_coords
    ],
}

RAW_OUTPUT_PATH.write_text(json.dumps(raw_payload, ensure_ascii=False, indent=2), encoding="utf-8")
MATCH_FRAMES_OUTPUT_PATH.write_text(
    json.dumps(match_frames_payload, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

print("wrote:", RAW_OUTPUT_PATH, f"({RAW_OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")
print("wrote:", MATCH_FRAMES_OUTPUT_PATH, f"({MATCH_FRAMES_OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")

## 7. Feed Into Trajectory Lab

After running the notebook, use the generated path directly:

```python
MATCH_FRAMES_PATH = CV_OUTPUT_DIR / f"{INPUT_VIDEO_PATH.stem}_ball_coords_match_frames.json"
```